In [ ]:
# Patient2Vec Demo

This notebook demonstrates a lightweight multimodal learning workflow for integrating genomics, transcriptomics, and clinical traits into a shared patient embedding.

## Workflow
- load synthetic multimodal patient data
- merge and tensorize the data
- train a small multimodal neural network
- inspect phenotype predictions
- visualize patient embeddings


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import load_data
from src.tensorization import tensorize
from src.model import PatientEncoder
from src.visualization import plot_embeddings


In [ ]:
## Load data


In [ ]:
df = load_data(
    "data/sample_genomics.csv",
    "data/sample_expression.csv",
    "data/sample_clinical.csv",
)

display(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


In [ ]:
## Tensorize multimodal inputs


g, r, c, y, clean_df = tensorize(df)
patient_ids = clean_df["PATIENT_ID"].tolist()

print("\nTensor shapes:")
print("Genomics:", g.shape)
print("Expression:", r.shape)
print("Clinical:", c.shape)
print("Labels:", y.shape)

display(clean_df.head())


## Initialize model


model = PatientEncoder(
    g_dim=g.shape[1],
    r_dim=r.shape[1],
    c_dim=c.shape[1],
    embedding_dim=8,
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

model


## Train the model


epochs = 300
history = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    embedding, logits = model(g, r, c)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        acc = (preds == y).float().mean().item()

    history.append({
        "epoch": epoch,
        "loss": loss.item(),
        "accuracy": acc
    })

    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Acc: {acc:.2f}")

history_df = pd.DataFrame(history)
display(history_df.tail())


## Training curves


plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["accuracy"])
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy")
plt.show()


## Inspect predictions


model.eval()
with torch.no_grad():
    embedding, logits = model(g, r, c)
    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    preds = (probs > 0.5).astype(int)

results = pd.DataFrame({
    "patient_id": patient_ids,
    "predicted_probability": probs,
    "predicted_label": preds,
    "true_label": y.squeeze().cpu().numpy().astype(int),
})

display(results)


## Visualize patient embeddings


Path("figures").mkdir(exist_ok=True)

plot_embeddings(
    embedding,
    y,
    patient_ids=patient_ids,
    save_path="figures/patient_embeddings.png",
)

print("Saved figure to: figures/patient_embeddings.png")


## Biological interpretation


summary = clean_df[["PATIENT_ID", "NPPA", "NPPB", "BMI", "AGE", "SBP", "HF_LABEL"]].copy()
summary = summary.sort_values(["HF_LABEL", "NPPA", "NPPB"], ascending=[False, False, False])
display(summary)


In this toy example, patients with higher **NPPA/NPPB** expression and less favorable clinical profiles align with the heart-failure-like label. In larger and more realistic datasets, the same workflow could be extended to multimodal patient stratification, disease subtyping, and representation learning across biological scales.
